# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library for reproducible data access and processing via its Croissant schema.

### Dataset Source
The dataset is sourced via a Croissant schema URL ([JSON-LD](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)).

In [ ]:
# Ensure `mlcroissant` is available
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant metadata and print main fields
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # Note: Do not subscript or iterate; use attributes.
print(f"Dataset Name: {meta.name}\nDescription: {meta.description}\nVersion: {meta.version}\nLicense: {meta.license}")

## 2. Data Overview
Review available record sets and their fields by `@id`.

We'll enumerate record sets, fields, and columns, referencing their `@id` as required by Croissant and the FAIR^2 protocol.

In [ ]:
# List record sets and their fields/columns with @id
def print_record_sets_and_fields(ds):
    print("Available Record Sets and their Fields:\n")
    for rs in ds.metadata.record_sets:
        print(f"Record Set: {rs.name} (@id: {rs.id})")
        if hasattr(rs, "fields") and rs.fields:
            for field in rs.fields:
                print(f"  Field: {field.name} (@id: {field.id}, datatype: {getattr(field, 'data_type', 'N/A')})")
        if hasattr(rs, "columns") and rs.columns:
            print("  Columns:")
            for c in rs.columns:
                print(f"    Column: {getattr(c, 'name', 'N/A')} (@id: {c.id if hasattr(c, 'id') else 'N/A'})")
        print()

print_record_sets_and_fields(dataset)

## 3. Data Extraction
Load data from a chosen record set into a DataFrame for analysis.

**Reference all entities by their `@id`!**

First, list record sets by `@id` for easy reference.

In [ ]:
# List of available record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Available record_set @ids:\n", record_set_ids)

# Load all (or a subset of) record sets into pandas DataFrames, referenced by @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {rs_id} with shape {dataframes[rs_id].shape}")
    except Exception as e:
        print(f"Warning: Could not load recordset {rs_id}: {e}")

In [ ]:
# Preview columns of a specific record set, e.g., proteins data, by its @id
# (Change this 'main_record_set_id' variable to the primary record set id from previous output)
main_record_set_id = record_set_ids[0]  # Change if needed
print(f"Columns for {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, grouping, etc., referencing all columns/fields by `@id`.

In [ ]:
# Example EDA: Filter, normalize, group by
df = dataframes[main_record_set_id]
# Select a likely numeric field. We'll print all columns for inspection first.
print(df.dtypes)
# Example: choose a numeric field looking like "MW (kDa)" or similar by its @id from schema (adjust as appropriate)
numeric_field_id = None
for c in df.columns:
    if 'mw' in c.lower() or 'weight' in c.lower() or 'abundance' in c.lower():
        numeric_field_id = c
        break
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include=['float', 'int']).columns[0] # fallback

print(f"Using numeric field: {numeric_field_id}")

# Filter records to those above a threshold
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records for {numeric_field_id} > {threshold:.2f}")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (z-score):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a likely categorical field (e.g. sample, accession, batch, etc.); adjust as needed.
demo_cats = ['sample', 'accession', 'description', 'batch']
group_field = None
for c in df.columns:
    if any([cat in c.lower() for cat in demo_cats]):
        group_field = c
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No group field identified in the columns.")

## 5. Visualization
Visualize distributions or relationships from the main record set.

> Note: You may need to adjust the field name(s) below based on outputs above and schema definitions.

In [ ]:
import matplotlib.pyplot as plt

# Simple histogram of the chosen numeric field
plt.figure(figsize=(8,4))
df[numeric_field_id].hist(bins=40)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field exists, show boxplot of numeric field per group
if group_field is not None:
    filtered_df.boxplot(column=numeric_field_id, by=group_field, rot=45, figsize=(10,5))
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to programmatically access, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. Referencing all data entities by their `@id` enables reproducible, schema-driven analysis. We:
- Loaded dataset metadata and records
- Investigated record sets, fields, and columns by their Croissant `@id`
- Extracted and inspected data in pandas
- Applied basic filtering, normalization, and grouping
- Visualized distributions for a key numeric field

You can build upon these steps for further feature engineering, advanced statistical analysis, and integration with machine learning workflows.